# Lab 4: More EDA - *Encodings, Color, and Chart Types*

## Objectives

By the end of this lab, you should be able to:

- Reshape data from wide to long with `.melt()`, and explain how it complements `.pivot_table()` from Lab 2
- Build bar charts, line charts, and histograms using pandas' built-in plotting, matplotlib, and seaborn, and know when to reach for each
- Build a heatmap that uses color to encode a value across two categorical dimensions

## Setup

In [ ]:
# !pip install pandas matplotlib seaborn plotly palmerpenguins

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from palmerpenguins import load_penguins

flights = pd.read_csv("flights.csv") #NOTE: You may need to change the path here to read in flights.csv.
penguins = load_penguins()

## 1. Reshaping data: wide to long with `.melt()`

In Lab 2, you used `.pivot_table()` to turn a long, grouped summary into a wide table, one that's easier for a person to scan. Let's rebuild that same wide table here.

In [5]:
delay_wide = flights.pivot_table(values="dep_delay", index="month", columns="origin", aggfunc="mean")
delay_wide

origin,EWR,JFK,LGA
month,,,
1,14.905748,8.615826,5.641560
2,13.067263,11.791355,6.961582
3,18.102457,10.721825,10.232041
4,17.400058,12.249059,11.508915
5,15.366141,12.519432,10.630508
6,22.470810,20.499729,19.296698
7,22.035112,23.769262,18.995163
8,13.493714,12.914358,11.244042
9,7.290954,6.635776,6.207439


This table is great for reading, but it's the wrong shape for plotting. Most plotting libraries, including seaborn and Plotly Express, expect **long** data: one row per observation, with the category (here, `origin`) stored as a value in a column, not spread across separate columns. To get back to that shape, we use `.melt()`.

`.melt()` is the reverse of `.pivot_table()`, with one important difference: `.pivot_table()` both aggregates and reshapes at once, while `.melt()` is a pure reshape. It doesn't compute anything new; it just unpivots columns back into rows.

In [ ]:
# melt the delay_wide DataFrame to create a long-format DataFrame with columns for month, origin, and avg_delay


<details>
<summary style="color: red; cursor: pointer;"><b>Click here to reveal the answer</b></summary>

### Solution Code

```python
delay_long = delay_wide.reset_index().melt(
    id_vars="month",       # column(s) to keep as-is
    var_name="origin",     # name for the new column holding the old column headers
    value_name="avg_delay" # name for the new column holding the values
)
delay_long.head()
```

</details>

`id_vars` tells `.melt()` which column(s) identify each row and should stay put; every other column gets stacked into two new columns, one holding the old column names (`var_name`) and one holding the values that used to live under those names (`value_name`). We'll use `delay_long` again in Section 3 below.

## 2. Bar charts

Bar charts are one of the most common ways to compare a value across categories. Let's look at three ways to build one, from quickest to most controllable.

**pandas' built-in `.plot()`** is the fastest way to get a bar chart directly from a Series or DataFrame, with no extra imports beyond pandas itself.

In [ ]:
# create a simple bar chart of species counts using the native plot()

<details>
<summary style="color: red; cursor: pointer;"><b>Click here to reveal the answer</b></summary>

### Solution Code

```python
species_counts = penguins["species"].value_counts()
species_counts.plot(kind="bar", title="Penguin counts by species")
plt.show()
```

</details>

Under the hood, pandas' `.plot()` is calling matplotlib for you. If you needed more control (say, custom colors or annotations), you'd drop down to matplotlib directly, using the `fig, ax` pattern from Lab 3:

In [ ]:
# create a simple bar chart of species counts using the native matplotlib

<details>
<summary style="color: red; cursor: pointer;"><b>Click here to reveal the answer</b></summary>

### Solution Code

```python
fig, ax = plt.subplots()
ax.bar(species_counts.index, species_counts.values)
ax.set_title("Penguin counts by species")
ax.set_ylabel("Count")
plt.show()
```

</details>

**seaborn** sits in between: it's DataFrame-aware like pandas' `.plot()`, but with more chart types and easier styling than matplotlib alone. It's also where color starts to do real work for us. Here, we compare average body mass across species, and use `hue` to bring in a *second* categorical variable, island, as color, rather than repeating the same information that's already on the x-axis.

In [ ]:
# create a grouped bar chart of average body mass by species and island using seaborn


<details>
<summary style="color: red; cursor: pointer;"><b>Click here to reveal the answer</b></summary>

### Solution Code

```python
fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(data=penguins, x="species", y="body_mass_g", hue="island", ax=ax)
ax.set_title("Average body mass by species and island")
ax.set_xlabel("Species")
ax.set_ylabel("Body mass (g)")
plt.show()
```

</details>

Notice what changed: position (the x-axis) encodes `species`, height encodes `body_mass_g`, and color encodes `island`. That's three variables in one chart, each carried by a different channel. Also notice some species/island combinations are missing entirely: not every species was observed on every island, and seaborn simply leaves that bar out rather than showing a misleading zero.

## 3. Line charts

Line charts are the standard choice when position along the x-axis represents an inherent order, most often time. Let's plot the `delay_long` data we built in Section 1: average departure delay by month, with one line per airport.

In [ ]:
#create a line chart of average departure delay by month, with hue set to origin


<details>
<summary style="color: red; cursor: pointer;"><b>Click here to reveal the answer</b></summary>

### Solution Code

```python
fig, ax = plt.subplots(figsize=(7, 5))
sns.lineplot(data=delay_long, x="month", y="avg_delay", hue="origin", marker="o", ax=ax)
ax.set_title("Average departure delay by month")
ax.set_xlabel("Month")
ax.set_ylabel("Average departure delay (minutes)")
plt.show()
```

</details>

Here, color again encodes a categorical variable, this time `origin`, letting us compare three trends on one chart instead of needing three separate charts. This is the same underlying idea as the loop-and-`ax.scatter()` pattern from Lab 3, where you built one series per group in a loop; `sns.lineplot()` with `hue=` does that looping for you.

## 4. Histograms

A histogram shows the distribution of a single numeric variable by dividing its range into bins and counting how many observations fall into each one.

In [ ]:
# Create a histogram of penguin body mass


<details>
<summary style="color: red; cursor: pointer;"><b>Click here to reveal the answer</b></summary>

### Solution Code

```python
penguins["body_mass_g"].plot(kind="hist", title="Distribution of penguin body mass")
plt.show()
```

</details>

seaborn's `sns.histplot()` gives more control, including an explicit `binwidth` and the ability to break the distribution down by group with color.

In [ ]:
# create the same histogram using seaborn with spedcies as hue


<details>
<summary style="color: red; cursor: pointer;"><b>Click here to reveal the answer</b></summary>

### Solution Code

```python
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(
    data=penguins, x="body_mass_g", hue="species",
    binwidth=200, alpha=0.6, element="step", ax=ax
)
ax.set_title("Distribution of penguin body mass by species")
ax.set_xlabel("Body mass (g)")
plt.show()
```

</details>

Here color separates the three species' distributions so you can compare their shapes directly, rather than only seeing the combined distribution from the first chart.

## 5. Heatmaps: encoding a value with color across two categories

So far, color has encoded a categorical variable directly. A **heatmap** uses color a little differently: it encodes a numeric value (often a count) across a grid formed by two categorical variables, using color intensity rather than position or height to show magnitude.

Let's build a heatmap of penguin counts by species and island. First, we need that count in a wide grid shape, which should look familiar from Section 1.

In [ ]:
# create a frequency table of species and island counts



<details>
<summary style="color: red; cursor: pointer;"><b>Click here to reveal the answer</b></summary>

### Solution Code

```python
species_island_counts = pd.crosstab(penguins["species"], penguins["island"])
species_island_counts
```

</details>

In [ ]:
# create a heatmap of the species and island counts using seaborn



<details>
<summary style="color: red; cursor: pointer;"><b>Click here to reveal the answer</b></summary>

### Solution Code

```python
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(species_island_counts, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Penguin counts by species and island")
plt.show()
```

</details>

`annot=True` prints the actual count inside each cell, so the color gives you an at-a-glance impression while the number gives you the exact value. `cmap="Blues"` sets a sequential colormap, appropriate here because count is a value that goes from "less" to "more"; a categorical variable would call for a qualitative palette instead, the way `hue=` automatically chose distinct colors for `species` and `island` earlier in this lab.




## On your own

### Part 1

Using the `fastfood` dataset (bundled with this lab, as in Lab 2):

1. Create a new column called `calorie_tier` by applying `pd.cut()` to `calories`, with `bins=3` and `labels=["Low", "Medium", "High"]`. (Reminder: `df["new_col"] = pd.cut(df["numeric_col"], bins=3, labels=[...])`.)
2. Build a `pivot_table` showing **average sodium**, with `restaurant` as rows and `calorie_tier` as columns.
3. `.melt()` that pivot table back into long format, so `restaurant` and `calorie_tier` are both columns again rather than one being spread across the table.
4. Using the long version, build a bar chart that compares average sodium by restaurant, with the `calorie_tier` mapped to color.

### Part 2

Using the `flights` data, make a heatmap, line plot, and histogram:
1. First, use `fig, ax = plt.subplots()` and add **three** axes, side by side, or top to bottom (i.e. `nrows=3` or `ncols=3`).
2. In the left/top axis: A histogram shows the distribution of a single Quantitative attribute (and multiple N attributes can be encoded as hue to make multiple histograms, as above). Choose a **quantitative attribute** to map to **horizontal position**. The y-axis should be frequency or count. Choose a **nominal attribute** to map to **hue**. Use **seaborn** to make overlapping histograms, as above.
3. In the middle axis: A lineplot is used for an **ordered** attribute and a **quantitative** attribute. Choose two attributes to build a lineplot with, and a **nominal attribute for the hue attribute**. You may subset the data to do this (for example, you could pick 5 interesting destinations instead of using all of them, and use dest for hue.)
4. In the right/bottom axis: Make a heatmap by choosing **two nominal (or ordinal) attributes**. Build a cross-tabulation with `pd.crosstab` as above. Then use this table as the input to a heatmap, where color represents the joint counts across the two categories. Pick a color map that is perceptually uniform, color blind friendly
5. For the plots, make sure you add labels, axes, tickmarks (as appropriate) and titles. 
6. For each plot, explain your choices: 
   1. How did you decide on the aspect ratio of each plot? 
   2. How did you pick attributes that made sense for each plot type?
   3. How did you decide on color use (color palette for hue, and colormap for quantitative data)?


## Submission

Export this notebook to HTML/PDF (`Jupyter: Export to HTML` from the Command Palette, or the "..." menu in the notebook toolbar) and submit the HTML file, as well as the ipynb file, on Canvas.